# MNIST demo with CREST 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST

In [3]:
import tensorflow as tf
import numpy as np

# Load data and define constants
xy,xy_test = tf.keras.datasets.mnist.load_data() 
xy_train   = [i[:50000]for i in xy]
xy_valid   = [i[50000:]for i in xy]
n_classes  = 10
patch_size = 28

In [4]:
len(xy)

2

#### Create the datasets

In [5]:
from crest.data.loading.StructuredDataset import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

#### Create the batchers

In [6]:
from crest.data.Batcher import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

#### Create the CREST graph

In [7]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

[None, None, None, None, None, None, None]


#### Build, train, and test using Model

In [8]:
from crest import Model

def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)

d = {
    'test' : tf.keras.metrics.SparseCategoricalAccuracy()
}

build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : ['sparce_categorical_accuracy', custom_metric]
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1
})

Traceback (most recent call last):
  File "/Users/ccruz/projects/terrahydro/development/crest/crest/utils/Metrics.py", line 35, in get_callbacks
    keras_avail = tf.keras.metrics.get(m)
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/metrics/__init__.py", line 181, in get
    return deserialize(str(identifier))
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/metrics/__init__.py", line 136, in deserialize
    return deserialize_keras_object(
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/utils/generic_utils.py", line 769, in deserialize_keras_object
    raise ValueError(
ValueError: Unknown metric function: sparce_categorical_accuracy. Please ensure this object is passed to the `custom_objects` argument. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

[]
Epoch 1/5
  1/250 [..............................] - ETA: 36s - loss: 71.7

2023-11-20 10:08:08.867283: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 2s 9ms/step - loss: 3.2148 - val_loss: 0.4592
Epoch 2/5
250/250 [==============================] - 2s 9ms/step - loss: 0.4799 - val_loss: 0.2707
Epoch 3/5
250/250 [==============================] - 2s 9ms/step - loss: 0.2505 - val_loss: 0.2172
Epoch 4/5
250/250 [==============================] - 2s 9ms/step - loss: 0.1575 - val_loss: 0.1869
Epoch 5/5
250/250 [==============================] - 2s 9ms/step - loss: 0.1148 - val_loss: 0.1599


In [9]:
# Evaluate the model on the test set
evaluate_kwargs= {
    'verbose' : 1,
    'return_dict' : True
}

result = model.evaluate(batch_test, verbose=1, return_dict=1)
print(result)

1/1 [==============================] - 0s 13ms/step - loss: 0.0481
{'loss': 0.048074282705783844}


Accuracy on the full test set should be > ~97%.

In [10]:
# Close the batchers
batch_train.close()
batch_valid.close()
batch_test.close()

### Make predictions

In [11]:
x_pred,_ = xy_test  # Use images from the testing set
ds_pred = StructuredDataset(*[x_pred], labels=[INP])
batch_pred = Batcher(ds_pred, **{
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
})

In [12]:
# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False
}

In [13]:
# Generate predictions on the whole test set
pred_kwargs['steps'] = 10000//pred_kwargs['batch_size']
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 
model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


Adding auxillary output like lat and lon when making a prediction

In [14]:
# Combine some fictous auxillary data with testing input for prediction
x,_ = xy_test
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_pred = (lat,lon,x)

In [15]:
# Create the prediction data and make predictions
ds_pred = StructuredDataset(*x_pred, labels=['lat','lon',INP])
aux_pred = model.predict(ds_pred,coords=['lat','lon'],**pred_kwargs)

In [16]:
# Check you have lat,lon now in your output
aux_pred 

{'class': array([[4.2541741e-19, 7.3117718e-32, 2.6668379e-20, ..., 1.0000000e+00,
         1.4736994e-14, 1.3549633e-14],
        [1.0390691e-17, 3.0825415e-13, 1.0000000e+00, ..., 4.2289910e-37,
         5.1538186e-19, 6.9073694e-33],
        [5.7448910e-06, 9.9999046e-01, 2.7686289e-08, ..., 1.0948625e-09,
         1.2712595e-06, 1.5366041e-08],
        ...,
        [2.2454177e-22, 6.5386722e-17, 1.5785142e-26, ..., 6.7711770e-13,
         7.7279516e-11, 2.1925095e-09],
        [5.0964714e-21, 8.2362250e-25, 6.6256498e-24, ..., 7.1223069e-27,
         2.1352214e-10, 9.1577093e-22],
        [2.7687677e-22, 6.9654483e-26, 1.7137162e-18, ..., 1.6834305e-27,
         3.6390379e-19, 1.4570239e-23]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}